# ritz 演示 notebook

> fitz, reforged in Rust.

本 notebook 演示 ritz 的核心 API：打开 PDF → 提取文本/图片/链接 → 渲染。

In [ ]:
import ritz
print('ritz version:', ritz.__version__)

## 1. 打开文档

In [ ]:
PDF = '../crates/mupdf/tests/samples/sample.pdf'  # 改成你自己的 PDF
doc = ritz.open(PDF)
print(f'页数: {len(doc)}')
print(f'加密: {doc.is_encrypted}')
print(f'format: {doc.metadata.get("format")}')
print(f'metadata: {doc.metadata}')

## 2. 页面属性

In [ ]:
page = doc[0]
print(f'rect:     {page.rect}')
print(f'mediabox: {page.mediabox}')
print(f'cropbox:  {page.cropbox}')
print(f'rotation: {page.rotation}°')

## 3. 文本提取（10 种模式）

ritz 与 PyMuPDF 完全兼容的 10 种 `get_text` 模式：

| 模式 | 返回类型 | 用途 |
|------|---------|------|
| `text` | str | 纯文本 |
| `html` / `xhtml` / `xml` | str | 结构化标记 |
| `json` / `rawjson` | str (JSON) | 序列化的 dict/rawdict |
| `words` | list[tuple] | 词级 bbox |
| `blocks` | list[tuple] | 块级 bbox |
| `dict` / `rawdict` | dict | 嵌套结构（rawdict 含逐字符 bbox）|

In [ ]:
text = page.get_text('text')
print(f'文本长度: {len(text)}')
print('前 200 字符:')
print(text[:200])

In [ ]:
# words 模式：每个词的 bbox
words = page.get_text('words')
print(f'词数: {len(words)}')
for w in words[:5]:
    x0, y0, x1, y1, txt = w[:5]
    print(f'  {txt!r:30s} bbox=({x0:.0f},{y0:.0f},{x1:.0f},{y1:.0f})')

In [ ]:
# dict 模式：嵌套 block → line → span
d = page.get_text('dict')
print(f'blocks: {len(d["blocks"])}')
if d['blocks']:
    b0 = d['blocks'][0]
    print(f'block[0]: type={b0["type"]}, bbox={tuple(round(x,1) for x in b0["bbox"])}, lines={len(b0.get("lines",[]))}')
    if b0.get('lines'):
        first_line = b0['lines'][0]
        if first_line['spans']:
            s0 = first_line['spans'][0]
            print(f'  span[0]: font={s0["font"]}, size={s0["size"]:.1f}, text={s0["text"]!r}')

In [ ]:
# rawdict 模式：每个 span 还有 chars 列表（含逐字符 bbox/origin）
rd = page.get_text('rawdict')
for b in rd['blocks']:
    for ln in b.get('lines', []):
        for s in ln.get('spans', []):
            if s.get('text'):
                print(f'span text={s["text"]!r}, chars={len(s["chars"])}')
                for c in s['chars'][:3]:
                    print(f'  c={c["c"]!r} bbox={tuple(round(x,1) for x in c["bbox"])} origin={tuple(round(x,1) for x in c["origin"])}')
                break
        else:
            continue
        break
    else:
        continue
    break

## 4. 图片提取

In [ ]:
# 注意：sample.pdf 可能无图。可换 text_graphic_image.pdf 演示
imgs = page.get_images()
print(f'页 {0} 图数: {len(imgs)}')
for i, img in enumerate(imgs):
    print(f'  [{i}] {img["width"]}x{img["height"]} {img["colorspace"]} bbox={tuple(round(x,0) for x in img["bbox"])}')

In [ ]:
# 含原始字节（include_data=True 时）
imgs_with_data = page.get_images(include_data=True)
for img in imgs_with_data:
    if 'image' in img:
        data = img['image']
        ext = img.get('ext', 'png')
        # JPEG/PNG/JPX 的原始压缩字节
        print(f'ext: {ext}, bytes: {len(data)}, magic: {data[:8].hex()}')

## 5. 链接

In [ ]:
links = page.get_links()
print(f'链接数: {len(links)}')
for link in links[:5]:
    print(f'  {link}')

## 6. 渲染 Pixmap

In [ ]:
pix = page.get_pixmap(dpi=144)
print(f'尺寸: {pix.width}x{pix.height}')
print(f'stride: {pix.stride}, components: {pix.components}')
print(f'samples 字节: {len(pix.samples)}')

In [ ]:
# 转 PNG bytes 并保存
png = pix.tobytes('png')
print(f'PNG 字节: {len(png)}, magic: {png[:8].hex()}')
with open('/tmp/ritz_demo.png', 'wb') as f:
    f.write(png)
print('saved to /tmp/ritz_demo.png')

In [ ]:
# 在 notebook 内联显示（可选）
from IPython.display import Image, display
display(Image(filename='/tmp/ritz_demo.png'))

## 7. 批量提取（单文档）

`get_text_batch()` 一次 FFI 调用拿全部页文本，避免 N 次循环。

In [ ]:
import time

t0 = time.perf_counter()
batch = doc.get_text_batch()
t1 = time.perf_counter()

t2 = time.perf_counter()
per_page = [doc[i].get_text('text') for i in range(len(doc))]
t3 = time.perf_counter()

print(f'batch:    {(t1-t0)*1000:.1f} ms')
print(f'per-page: {(t3-t2)*1000:.1f} ms')
print(f'内容一致: {batch == per_page}')

## 8. 并行处理多文档

`ritz.process_documents(paths)` 用 rayon 并行，无需 multiprocessing。

In [ ]:
import glob
# 找几个 PDF 做演示
paths = sorted(glob.glob('../vendor/mupdf/thirdparty/**/*.pdf', recursive=True))[:6]
print(f'测试 {len(paths)} 个文档')

t0 = time.perf_counter()
serial = []
for p in paths:
    try:
        d = ritz.open(p)
        serial.append(d.get_text_batch())
    except Exception:
        serial.append(None)
t1 = time.perf_counter()

parallel = ritz.process_documents(paths)
t2 = time.perf_counter()

print(f'串行: {(t1-t0)*1000:.0f} ms')
print(f'并行: {(t2-t1)*1000:.0f} ms')
print(f'加速: {((t1-t0)/(t2-t1)):.2f}x')
print(f'结果一致: {serial == parallel}')

## 9. 性能对比 vs PyMuPDF

如果安装了 PyMuPDF，可以做并排对比：

In [ ]:
try:
    import fitz
    has_fitz = True
except ImportError:
    has_fitz = False
    print('PyMuPDF 未安装，跳过对比')

if has_fitz:
    # 文档打开
    N = 50
    t0 = time.perf_counter()
    for _ in range(N):
        d = ritz.open(PDF); _ = len(d)
    t1 = time.perf_counter()
    for _ in range(N):
        d = fitz.open(PDF); _ = len(d); d.close()
    t2 = time.perf_counter()
    
    ritz_ms = (t1-t0)/N*1000
    fitz_ms = (t2-t1)/N*1000
    print(f'文档打开: ritz={ritz_ms:.2f}ms fitz={fitz_ms:.2f}ms ({fitz_ms/ritz_ms:.1f}x)')
    
    # 文本提取
    rd = ritz.open(PDF); rp = rd[0]
    fd = fitz.open(PDF); fp = fd[0]
    t0 = time.perf_counter()
    for _ in range(N):
        _ = rp.get_text('text')
    t1 = time.perf_counter()
    for _ in range(N):
        _ = fp.get_text('text')
    t2 = time.perf_counter()
    ritz_ms = (t1-t0)/N*1000
    fitz_ms = (t2-t1)/N*1000
    print(f'文本提取: ritz={ritz_ms:.3f}ms fitz={fitz_ms:.3f}ms ({fitz_ms/ritz_ms:.1f}x)')

## 进一步阅读

- README: [../README.md](../README.md)
- 性能报告: [../benchmarks/benchmark_report.md](../benchmarks/benchmark_report.md)
- 测试套件: [../python/tests/test_ritz.py](../python/tests/test_ritz.py)